# EEG · 02 · Retrieval ablation (Experiment 2)
**Question:** does the improvement depend on using the *correct* EEG?

The decisive control: `correct` must clearly beat `permuted` and `zero`.

In [ ]:
# Run from the PROJECT ROOT so relative paths (configs/, data/, outputs/)
# resolve exactly like the scripts do.
import sys, os
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'configs')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
print('project root:', os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_config, get_device
from src.data import build_datamodule
from src.models import build_model_from_checkpoint
from src.evaluation import evaluate_ablation, conclusion_from_summary, topk_candidates

cfg = load_config('configs/EEG/exp02_retrieval_ablation.yaml')
device = get_device(cfg.get('runtime.device','auto'))
dm = build_datamodule(cfg).prepare()
ckpt = 'outputs/exp01_eeg_to_clip/checkpoints/best.pt'
model, _ = build_model_from_checkpoint(cfg, ckpt, device, dm.voxel_counts)

In [ ]:
res = evaluate_ablation(model, cfg, dm, split='test',
                        conditions=['correct','permuted','zero'], device=device)
summary = res['summary']
summary[summary.subject_id=='all'].pivot_table(index='metric_name', columns='condition', values='value')

## Top-k by condition

In [ ]:
piv = summary[(summary.subject_id=='all') & (summary.metric_name.str.startswith('retrieval/top'))]
piv = piv.pivot_table(index='metric_name', columns='condition', values='value')
piv.plot(kind='bar', figsize=(8,4)); plt.ylabel('accuracy'); plt.title('Top-k by condition'); plt.show()
piv

## Qualitative retrieval (correct condition)

In [ ]:
pred = res['predictions']['correct'][dm.subjects[0]]
cand = topk_candidates(pred['clip_pred'], pred['clip_target'], k=5)
from src.data import load_image
paths_ = pred['image_paths']
n_show = min(4, len(paths_))
fig, axes = plt.subplots(n_show, 6, figsize=(14, 2.4*n_show))
for r in range(n_show):
    axes[r,0].imshow(load_image(paths_[r])); axes[r,0].set_title('query', fontsize=8); axes[r,0].axis('off')
    for j, ci in enumerate(cand[r]):
        axes[r,j+1].imshow(load_image(paths_[ci]))
        axes[r,j+1].set_title(('hit' if ci==r else '')+f'#{j+1}', fontsize=8); axes[r,j+1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
concl = conclusion_from_summary(summary, metric='retrieval/top5', subject='all')
print(concl['message']); concl

**Takeaway:** if `correct` does not clearly exceed `permuted`/`zero`, we must NOT claim the model uses real brain information — report it honestly. For EEG, evaluation is at the image level (mean over repetitions).